In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
import pickle
import os

In [3]:
# --- 1. Konfigurasi dan Pemuatan Data ---
# Pastikan Anda telah mengunduh 'ml-latest-small' dari: https://grouplens.org/datasets/movielens/
# dan letakkan di folder yang sama dengan script ini.
DATA_DIR = 'ml-25m/'
MODEL_DIR = 'models/'

In [4]:
def load_data():
    """
    Memuat data movies dan ratings dari folder MovieLens.
    """
    print("Memuat data...")
    try:
        movies_df = pd.read_csv(os.path.join(DATA_DIR, 'movies.csv'))
        ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'ratings.csv'))
        print("Data berhasil dimuat.")
        return movies_df, ratings_df
    except FileNotFoundError:
        print(f"Error: Folder '{DATA_DIR}' tidak ditemukan.")
        print("Silakan unduh dataset 'ml-latest-small' dari GroupLens dan letakkan di folder yang benar.")
        return None, None

In [5]:
def create_content_based_model(movies_df):
    """
    Melatih model Content-Based menggunakan TF-IDF pada genre.
    Menyimpan model (vectorizer dan matrix similarity) ke file pickle.
    """
    print("Memulai pelatihan model Content-Based...")
    # Mengganti '|' dengan spasi agar TF-IDF dapat memproses genre
    movies_df['genres_cleaned'] = movies_df['genres'].str.replace('|', ' ')
    
    # Membuat TF-IDF Vectorizer
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movies_df['genres_cleaned'])
    
    # Menghitung Cosine Similarity Matrix
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    print("Model Content-Based (Cosine Similarity) berhasil dibuat.")
    
    # Menyimpan model dan data yang diperlukan
    pickle.dump(cosine_sim, open(os.path.join(MODEL_DIR, 'content_cosine_sim.pkl'), 'wb'))
    pickle.dump(tfidf, open(os.path.join(MODEL_DIR, 'content_tfidf_vectorizer.pkl'), 'wb'))
    
    # Kita juga menyimpan movies_df yang sudah bersih untuk digunakan di Streamlit
    pickle.dump(movies_df, open(os.path.join(MODEL_DIR, 'movies_df.pkl'), 'wb'))
    print("Model Content-Based berhasil disimpan.")
    return movies_df, cosine_sim

In [6]:
def create_collaborative_filtering_model(ratings_df):
    """
    Melatih model Collaborative Filtering menggunakan SVD (Surprise).
    Menyimpan model ke file pickle.
    """
    print("Memulai pelatihan model Collaborative Filtering (SVD)...")
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(ratings_df[['userId', 'movieId', 'rating']], reader)
    
    # Kita akan melatih pada seluruh dataset untuk production
    # (Untuk evaluasi, gunakan train_test_split dari Surprise)
    trainset = data.build_full_trainset()
    
    # Menggunakan SVD
    algo = SVD(n_factors=100, n_epochs=20, random_state=42)
    algo.fit(trainset)
    print("Model SVD berhasil dilatih.")
    
    # Menyimpan model SVD
    pickle.dump(algo, open(os.path.join(MODEL_DIR, 'svd_model.pkl'), 'wb'))
    print("Model SVD berhasil disimpan.")
    return algo

In [7]:
def get_content_based_recommendations(liked_movie_titles, movies_df, cosine_sim, top_n=50):
    """
    Memberikan rekomendasi content-based berdasarkan 5 film yang disukai.
    """
    # Membuat series untuk mapping judul ke index
    indices = pd.Series(movies_df.index, index=movies_df['title'])
    
    # Mendapatkan index dari 5 film yang disukai
    liked_indices = [indices[title] for title in liked_movie_titles if title in indices]
    
    if not liked_indices:
        return pd.DataFrame() # Kembalikan DataFrame kosong jika tidak ada film yang cocok

    # Menghitung rata-rata skor similaritas dari 5 film tersebut
    avg_sim_scores = np.zeros(cosine_sim.shape[0])
    for idx in liked_indices:
        avg_sim_scores += cosine_sim[idx]
        
    avg_sim_scores /= len(liked_indices)
    
    # Mengurutkan film berdasarkan skor similaritas rata-rata
    sim_scores = list(enumerate(avg_sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Mendapatkan skor dari top_n film
    sim_scores = sim_scores[1:top_n+len(liked_indices)] # +len untuk skip film yg sudah disukai
    
    # Mendapatkan index film
    movie_indices = [i[0] for i in sim_scores]
    
    # Mengambil judul film dan skor
    recs_df = movies_df.iloc[movie_indices][['title', 'genres']]
    recs_df['score'] = [i[1] for i in sim_scores]
    
    # Menghapus film yang sudah disukai dari rekomendasi
    recs_df = recs_df[~recs_df['title'].isin(liked_movie_titles)]
    
    return recs_df.head(top_n)

def get_collaborative_filtering_recommendations(user_id, svd_model, movies_df, ratings_df, top_n=50):
    """
    Memberikan rekomendasi collaborative filtering (SVD) untuk user_id.
    """
    # Mendapatkan daftar film yang belum ditonton oleh user
    movie_ids_watched = ratings_df[ratings_df['userId'] == user_id]['movieId'].unique()
    movie_ids_all = movies_df['movieId'].unique()
    movie_ids_to_predict = np.setdiff1d(movie_ids_all, movie_ids_watched)
    
    # Membuat test set untuk prediksi
    testset = [[user_id, movie_id, 4.] for movie_id in movie_ids_to_predict]
    
    # Memprediksi rating
    predictions = svd_model.test(testset)
    
    # Mengurutkan prediksi berdasarkan estimasi rating
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # Mengambil top_n rekomendasi
    recs = []
    for pred in predictions[:top_n]:
        movie_id = pred.iid
        movie_title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
        movie_genre = movies_df[movies_df['movieId'] == movie_id]['genres'].values[0]
        recs.append({'title': movie_title, 'genres': movie_genre, 'score': pred.est})
        
    return pd.DataFrame(recs)

def get_hybrid_recommendations(user_id, liked_movie_titles, movies_df, cosine_sim, svd_model, ratings_df, top_n=10):
    """
    Menggabungkan rekomendasi Content-Based dan Collaborative Filtering.
    """
    print(f"Membuat rekomendasi hibrida untuk User {user_id}...")
    
    # 1. Dapatkan Rekomendasi Content-Based
    cb_recs = get_content_based_recommendations(liked_movie_titles, movies_df, cosine_sim, top_n=50)
    
    # 2. Dapatkan Rekomendasi Collaborative Filtering
    cf_recs = get_collaborative_filtering_recommendations(user_id, svd_model, movies_df, ratings_df, top_n=50)
    
    if cb_recs.empty or cf_recs.empty:
        print("Salah satu model tidak menghasilkan rekomendasi. Mengembalikan hasil dari model yang ada.")
        return cb_recs.head(top_n) if not cb_recs.empty else cf_recs.head(top_n)

    # 3. Hybrid: Re-ranking
    # Normalisasi skor (Skor CF: 0.5-5, Skor CB: 0-1)
    cb_recs['score_norm'] = cb_recs['score']
    cf_recs['score_norm'] = cf_recs['score'] / 5.0
    
    # Gabungkan (Merge) berdasarkan judul
    hybrid_df = pd.merge(cb_recs, cf_recs, on='title', suffixes=('_cb', '_cf'), how='outer')
    
    # Isi NaN dengan 0 (jika film hanya ada di satu list)
    hybrid_df = hybrid_df.fillna(0)
    
    # Hitung skor hibrida (simple weighted average, 50/50)
    hybrid_df['hybrid_score'] = (hybrid_df['score_norm_cb'] * 0.5) + (hybrid_df['score_norm_cf'] * 0.5)
    
    # Ambil genre dari salah satu kolom (pastikan tidak 0)
    hybrid_df['genres'] = hybrid_df['genres_cb'].where(hybrid_df['genres_cb'] != 0, hybrid_df['genres_cf'])
    
    # Urutkan berdasarkan skor hibrida
    hybrid_df = hybrid_df.sort_values(by='hybrid_score', ascending=False)
    
    # Filter kolom
    final_recs = hybrid_df[['title', 'genres', 'hybrid_score', 'score_norm_cb', 'score_norm_cf']]
    
    return final_recs.head(top_n)

def get_recommendation_explanation(recommended_movie_title, liked_movie_titles, movies_df, cosine_sim):
    """
    Membuat penjelasan (XAI) mengapa sebuah film direkomendasikan.
    Fokus pada kesamaan genre (Content-Based).
    """
    # Mapping judul ke index
    indices = pd.Series(movies_df.index, index=movies_df['title'])
    
    if recommended_movie_title not in indices:
        return "Alasan tidak ditemukan."
        
    rec_idx = indices[recommended_movie_title]
    
    # Cari film yang disukai yang paling mirip
    best_match_title = None
    best_sim_score = -1.0
    
    for title in liked_movie_titles:
        if title in indices:
            liked_idx = indices[title]
            sim = cosine_sim[rec_idx][liked_idx]
            if sim > best_sim_score:
                best_sim_score = sim
                best_match_title = title
                
    if best_match_title:
        # Dapatkan genre dari kedua film
        rec_genres = set(movies_df.loc[rec_idx]['genres'].split('|'))
        match_genres = set(movies_df.loc[indices[best_match_title]]['genres'].split('|'))
        
        # Cari irisan genre
        common_genres = list(rec_genres.intersection(match_genres))
        
        if common_genres:
            return f"Karena Anda menyukai **{best_match_title}**, kami merekomendasikan film ini yang juga memiliki genre: **{', '.join(common_genres)}**."
        else:
            return f"Rekomendasi ini mirip dengan selera pengguna lain yang juga menyukai **{best_match_title}**."
    
    return "Film ini direkomendasikan berdasarkan tren pengguna dengan selera yang mirip dengan Anda."


# --- 5. Main Execution ---
def main():
    """
    Fungsi utama untuk menjalankan seluruh pipeline ETL dan training.
    """
    print("Menjalankan pipeline training...")
    
    # Langkah 1: Muat Data
    movies_df, ratings_df = load_data()
    
    if movies_df is None or ratings_df is None:
        return
        
    # Langkah 2: Latih dan Simpan Model Content-Based
    # Kita perlu menyimpan movies_df yang sudah di-update
    movies_df, cosine_sim = create_content_based_model(movies_df.copy())
    
    # Langkah 3: Latih dan Simpan Model Collaborative Filtering
    svd_model = create_collaborative_filtering_model(ratings_df.copy())
    
    print("\n--- Pipeline Selesai ---")
    print(f"Model berhasil disimpan di folder '{MODEL_DIR}'.")
    print("Anda sekarang dapat menjalankan 'streamlit run app.py'")
    
    # --- Contoh Pengujian (Opsional) ---
    print("\n--- Menjalankan Pengujian Cepat ---")
    
    # Ambil user ID dan 5 film yang disukainya
    TEST_USER_ID = 1
    user_ratings = ratings_df[ratings_df['userId'] == TEST_USER_ID].sort_values('rating', ascending=False)
    liked_movie_ids = user_ratings.head(5)['movieId'].values
    liked_movie_titles = movies_df[movies_df['movieId'].isin(liked_movie_ids)]['title'].tolist()
    
    if not liked_movie_titles:
        print("User 1 belum memiliki 5 rating. Menggunakan film default.")
        liked_movie_titles = ['Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)', 'Waiting to Exhale (1995)', 'Heat (1995)']

    print(f"User {TEST_USER_ID} menyukai: {liked_movie_titles}")

    # Dapatkan rekomendasi hibrida
    recommendations = get_hybrid_recommendations(
        TEST_USER_ID, 
        liked_movie_titles, 
        movies_df, 
        cosine_sim, 
        svd_model, 
        ratings_df
    )
    
    print("\nTop 10 Rekomendasi Hibrida:")
    print(recommendations)
    
    # Dapatkan penjelasan untuk rekomendasi pertama
    if not recommendations.empty:
        top_rec_title = recommendations.iloc[0]['title']
        explanation = get_recommendation_explanation(top_rec_title, liked_movie_titles, movies_df, cosine_sim)
        print(f"\nPenjelasan untuk '{top_rec_title}':")
        print(explanation)

if __name__ == "__main__":
    main()


Menjalankan pipeline training...
Memuat data...
Data berhasil dimuat.
Memulai pelatihan model Content-Based...
Model Content-Based (Cosine Similarity) berhasil dibuat.
Model Content-Based berhasil disimpan.
Memulai pelatihan model Collaborative Filtering (SVD)...
Model SVD berhasil dilatih.
Model SVD berhasil disimpan.

--- Pipeline Selesai ---
Model berhasil disimpan di folder 'models/'.
Anda sekarang dapat menjalankan 'streamlit run app.py'

--- Menjalankan Pengujian Cepat ---
User 1 menyukai: ['Pulp Fiction (1994)', 'Saragossa Manuscript, The (Rekopis znaleziony w Saragossie) (1965)', 'Eternal Sunshine of the Spotless Mind (2004)', 'Dolce Vita, La (1960)', 'Dolls (2002)']
Membuat rekomendasi hibrida untuk User 1...

Top 10 Rekomendasi Hibrida:
                                             title              genres  \
15  Burnt by the Sun (Utomlyonnye solntsem) (1994)               Drama   
2                           American Beauty (1999)       Drama|Romance   
64                   